<div style="
    background-color:#2c3e50; 
    color:#ecf0f1; 
    font-weight:bold; 
    padding:20px 30px; 
    font-size:20px; 
    border-radius:8px; 
    text-align:center;
    font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
    letter-spacing: 0.5px;
">
    Household Missing Files
</div>

<h3 style="
    font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
    color:#D4495e;
    margin-top:15px;
    font-weight:600;
">
    Completeness of accelerometer data files for each monitored laundry appliance. The Total Expected column
indicates the number of accelerometer recordings that should have been collected, one for each main experiment file. Found
counts the number of accelerometer files actually present and containing valid data, while Missing and Missing % quantify
cases where the entire accelerometer file is absent.
</h3>

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# Load dataset
aggregated = pd.read_csv("aggregated_data.csv")

# Filter rows where environment == 'household'
household = aggregated[aggregated["environment"] == "household"]

# Collect all accelerometer files under ./acc/household
acc_files = []
for root, dirs, files in os.walk("./acc/household"):
    for file in files:
        if file.endswith(".parquet"):
            acc_files.append(os.path.join(root, file))

In [ ]:
remaining_files = acc_files.copy()

for idx, row in household.iterrows():
    partial = row["file_name"].split(".")[0]

    # Find matches in the remaining files
    matches = [f for f in remaining_files if partial in f]

    if not matches:
        print(f"❌ Partial: {partial}, No matches found.")
        continue

    for m in matches: # Optimization to reduce search space
        remaining_files.remove(m)

    try:
        acc_df = pd.read_parquet(matches[0])

        if not acc_df.empty:
            print(f"✅ Partial: {partial}, Matches: {matches}", idx)
        else:
            print(f"❌ Partial: {partial}, Matches: {matches}", idx)

    except Exception as e:
        print(f"❌ Partial: {partial} Error processing.")
        print(f"Error: {e}")

In [ ]:
import pandas as pd
from collections import defaultdict

stats = defaultdict(lambda: {"total": 0, "found": 0, "model": None})

for _, row in household.iterrows():
    partial = row["file_name"].rsplit(".", 1)[0]                    # drop extension
    brand    = partial.split("_", 1)[0]
    model_only = partial.split("_", 1)[1].split("_")[0]             # e.g. WF80F5E0W2W
    brand_model = f"{brand}_{model_only}"                           # key

    stats[brand_model]["total"] += 1
    stats[brand_model]["model"] = model_only

    matches = [f for f in acc_files if partial in f]
    if not matches:
        continue

    try:
        acc_df = pd.read_parquet(matches[0])
        if not acc_df.empty:
            stats[brand_model]["found"] += 1
    except Exception:
        pass

summary_df = pd.DataFrame([
    {
        "Brand": machine.split("_")[0],                       # e.g. SAMSUNG_WF80F5E0W2W
        "Model": vals["model"],                   # e.g. WF80F5E0W2W
        "Total Expected": vals["total"],
        "Existing": vals["found"],
        "Missing": vals["total"] - vals["found"],
        "Missing %": 100 * (vals["total"] - vals["found"]) / vals["total"],
    }
    for machine, vals in stats.items()
])

def get_house_code(model):
    if "WDYN654D" in model:
        return "1"
    elif "WW80T554DTW" in model:
        return "2"
    elif "EWF1272EOW" in model:
        return "3"
    elif "FMG823B" in model:
        return "4"
    elif "L6FBG141" in model:
        return "5"
    elif "WF80F5E0W2W" in model:
        return "6"
    elif "3TS866EE" in model:
        return "7"
    elif "DHS7412PA0" in model:
        return "6"
    else:
        return None

def get_house_type(model):
    if "WDYN654D" in model:
        return "Wm/Dr"
    elif "WW80T554DTW" in model:
        return "Wm"
    elif "EWF1272EOW" in model:
        return "Wm"
    elif "FMG823B" in model:
        return "Wm"
    elif "L6FBG141" in model:
        return "Wm"
    elif "WF80F5E0W2W" in model:
        return "Wm"
    elif "3TS866EE" in model:
        return "Wm"
    elif "DHS7412PA0" in model:
        return "Dr"
    else:
        return None

# Build the ID column
summary_df["House ID"] = summary_df["Model"].apply(get_house_code)
summary_df["Type"] = summary_df["Model"].apply(get_house_type)

# Helpers for sorting
id_ser = summary_df["House ID"].fillna("")

# type: WM first, then DR, unknown last
type_order = {"WM": 0, "DR": 0}
summary_df["_type"] = (
    id_ser.str.extract(r"^(WM|DR)", expand=False).map(type_order).fillna(2)
)

# number after the letter(s), e.g., "H6" -> 6
summary_df["_num"] = (
    id_ser.str.extract(r"(\d+)", expand=False).astype(float)
)

# Sort and clean up
summary_df = (
    summary_df
      .sort_values(by=["_type", "_num", "Brand"])
      .drop(columns=["_type", "_num"])
)

# Optional: reorder columns
summary_df = summary_df[["House ID","Type", "Brand","Model", "Total Expected", "Existing", "Missing", "Missing %"]]
print(summary_df)



<h3 style="
    font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
    color:#D4495e;
    margin-top:15px;
    font-weight:600;
">
    Within-file completeness of accelerometer recordings for each monitored laundry appliance, assuming a sampling
rate of 200 Hz. Expected rows are calculated from each file’s start–end duration and nominal sampling frequency. Actual rows
are the number of unique samples present. Missing rows and Missing % quantify missing samples within existing files
(excluding entirely missing files).
</h3>

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict

FS = 200.0  # Hz

# --- helper: count missing samples given a time index at fixed fs ---
def missing_samples_by_duration(idx: pd.Index, fs: float = FS):
    """
    Returns dict(missing, actual, expected, start, end) using duration-based logic:
    expected ≈ round((end-start)/median_dt) + 1, robust to jitter.
    """
    if isinstance(idx, pd.Series):
        t = pd.to_datetime(idx, utc=True, errors="coerce").dropna().sort_values().drop_duplicates()
    else:
        if not isinstance(idx, pd.DatetimeIndex):
            idx = pd.to_datetime(idx, utc=True, errors="coerce")
        t = pd.DatetimeIndex(idx).tz_convert(None).sort_values().unique()

    n = len(t)
    if n == 0:
        return {"missing": 0, "actual": 0, "expected": 0, "start": None, "end": None}
    if n == 1:
        return {"missing": 0, "actual": 1, "expected": 1, "start": t[0], "end": t[0]}

    # duration in seconds
    dur_sec = (t[-1] - t[0]).total_seconds()

    # robust period estimate from data (median Δt), clipped to reasonable bounds around 1/fs
    diffs_ns = np.diff(t.asi8)  # nanoseconds
    diffs_s = diffs_ns / 1e9
    # guard against big outliers
    p99 = np.percentile(diffs_s, 99)
    diffs_s_clipped = np.clip(diffs_s, 0, p99)
    median_dt = float(np.median(diffs_s_clipped)) if diffs_s_clipped.size else (1.0 / fs)

    # sanity: if median_dt way off, fall back to nominal
    if not (0.5 / fs <= median_dt <= 1.5 / fs):
        median_dt = 1.0 / fs

    expected = int(round(dur_sec / median_dt)) + 1  # inclusive endpoints
    actual = int(n)
    missing = max(expected - actual, 0)

    return {"missing": missing, "actual": actual, "expected": expected, "start": t[0], "end": t[-1]}


# --- aggregate per Brand+Model over existing ACC files ---
acc_per_machine = defaultdict(lambda: {
    "brand": None, "model": None,
    "files": 0, "expected": 0, "actual": 0, "missing": 0
})

for _, row in household.iterrows():
    partial = row["file_name"].rsplit(".", 1)[0]
    brand = partial.split("_", 1)[0]
    model = partial.split("_", 1)[1].split("_")[0]
    key = f"{brand}_{model}"

    # find matching accelerometer parquet
    matches = [f for f in acc_files if partial in f]
    if not matches:
        continue  # entirely-missing ACC file is handled in your other table

    try:
        acc_df = pd.read_parquet(matches[0])
    except Exception:
        continue  # unreadable -> skip

    if acc_df is None or acc_df.empty:
        continue

    # prefer DatetimeIndex; otherwise try common timestamp column names
    if isinstance(acc_df.index, pd.DatetimeIndex):
        res = missing_samples_by_duration(acc_df.index, fs=FS)
    else:
        # try to locate a timestamp-like column
        cand = [c for c in acc_df.columns if str(c).lower() in {"timestamp","time","datetime","ts","date_time"}]
        if not cand:
            continue
        res = missing_samples_by_duration(acc_df[cand[0]], fs=FS)

    if res["expected"] == 0:
        continue

    s = acc_per_machine[key]
    s["brand"], s["model"] = brand, model
    s["files"]   += 1
    s["expected"] += int(res["expected"])
    s["actual"]   += int(res["actual"])
    s["missing"]  += int(res["missing"])

# --- build summary table (accelerometer, within-file, per 200 Hz samples) ---
acc_within_summary = (
    pd.DataFrame([
        {
            "Brand": v["brand"],
            "Model": v["model"],
            "Expected rows": v["expected"],
            "Actual rows": v["actual"],
            "Missing rows": v["missing"],
            "Missing %": round((v["missing"] / v["expected"]) * 100, 2) if v["expected"] else 0.0
        }
        for v in acc_per_machine.values()
    ])
    .sort_values(["Brand", "Model"])
    .reset_index(drop=True)
)

print(acc_within_summary)

# --- overall totals (accelerometer, existing files only) ---
tot_expected = int(acc_within_summary["Expected rows"].sum())
tot_actual   = int(acc_within_summary["Actual rows"].sum())
tot_missing  = int(acc_within_summary["Missing rows"].sum())
tot_pct      = round((tot_missing / tot_expected) * 100, 2) if tot_expected else 0.0

print("=== Overall ACC completeness (existing files only, 200 Hz samples) ===")
print(f"Total expected samples : {tot_expected:,}")
print(f"Total actual samples   : {tot_actual:,}")
print(f"Total missing samples  : {tot_missing:,}")
print(f"Total missing %        : {tot_pct:.2f}%")


<h3 style="
    font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
    color:#D4495e;
    margin-top:15px;
    font-weight:600;
">
    Summary of missing main operational household data due to full acquisition dropouts for each monitored laundry
appliance, assuming one row per second. Expected rows are computed from each file’s start–end duration. Actual rows are the
number of unique seconds present. Missing rows and Missing % quantify seconds absent within existing files (not counting
entirely missing files).
</h3>

In [ ]:
import pandas as pd
from collections import defaultdict

# --- helper (from earlier, already fixed) ---
def missing_rows_per_second(ts: pd.Series):
    sec = pd.to_datetime(ts, utc=True, errors="coerce").dropna()
    if sec.empty:
        return {"missing": 0, "actual": 0, "expected": 0, "start": None, "end": None}

    sec = sec.sort_values().dt.floor("s").drop_duplicates()
    start, end = sec.iloc[0], sec.iloc[-1]
    expected = int((end - start).total_seconds()) + 1
    actual = int(sec.size)
    missing = max(expected - actual, 0)
    return {"missing": missing, "actual": actual, "expected": expected, "start": start, "end": end}

# --- aggregate per machine ---
per_machine = defaultdict(lambda: {
    "brand": None, "model": None,
    "files": 0, "expected": 0, "actual": 0, "missing": 0
})

for _, row in household.iterrows():
    partial = row["file_name"].rsplit(".", 1)[0]
    brand = partial.split("_", 1)[0]
    model = partial.split("_", 1)[1].split("_")[0]
    key = f"{brand}_{model}"

    # Only process files that exist (presence handled elsewhere)
    try:
        # parse timestamp column; no infer_datetime_format needed
        df = pd.read_csv(row["file_path"], usecols=["timestamp"], parse_dates=["timestamp"])
    except Exception:
        continue

    res = missing_rows_per_second(df["timestamp"])
    if res["expected"] == 0:
        continue

    s = per_machine[key]
    s["brand"], s["model"] = brand, model
    s["files"] += 1
    s["expected"] += res["expected"]
    s["actual"]   += res["actual"]
    s["missing"]  += res["missing"]

within_sec_summary = (
    pd.DataFrame([
        {
            "Brand": v["brand"],
            "Model": v["model"],
            # "Files (present)": v["files"],
            "Expected rows": v["expected"],
            "Actual rows": v["actual"],
            "Missing rows": v["missing"],
            "Missing %": round((v["missing"] / v["expected"]) * 100, 2) if v["expected"] else 0.0
        }
        for v in per_machine.values()
    ])
    .sort_values(["Brand", "Model"])
    .reset_index(drop=True)
)

print(within_sec_summary)

# --- grand totals from the per-machine table ---
tot_expected = int(within_sec_summary["Expected rows"].sum())
tot_actual   = int(within_sec_summary["Actual rows"].sum())
tot_missing  = int(within_sec_summary["Missing rows"].sum())
tot_pct      = round((tot_missing / tot_expected) * 100, 2) if tot_expected else 0.0

print("=== Overall per-second completeness (existing files only) ===")
print(f"Total expected seconds : {tot_expected:,}")
print(f"Total actual seconds   : {tot_actual:,}")
print(f"Total missing seconds  : {tot_missing:,}")
print(f"Total missing %        : {tot_pct:.2f}%")

<div style="
    background-color:#2c3e50; 
    color:#ecf0f1; 
    font-weight:bold; 
    padding:20px 30px; 
    font-size:20px; 
    border-radius:8px; 
    text-align:center;
    font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
    letter-spacing: 0.5px;
">
    Laboratory missing data
</div>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

# Load dataset
aggregated = pd.read_csv('aggregated_data.csv')

# Find in agg_excel the column with that contains environment = laboratory
# laboratory = agg_excel[agg_excel['environment'] == 'laboratory']
laboratory = aggregated[
    (aggregated['environment'] == 'laboratory') &
    (aggregated['type'] == 'wm')
]

print(laboratory.head())

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict

ELECTRICAL_COLS_CANON = ["current", "frequency", "power", "power_factor", "voltage"]

def _per_file_feature_missing_and_expected_twopass(df: pd.DataFrame) -> dict:
    cols = df.columns
    n_rows = len(df)

    # --- Structural expected (like your original) ---
    expected = pd.Series(n_rows, index=cols, dtype="int64")

    nan_counts = df.isna().sum().reindex(cols).fillna(0).astype("int64")
    all_nan_cols = nan_counts.index[nan_counts == n_rows]

    all_minus1_cols = []
    for col in cols:
        s = pd.to_numeric(df[col], errors="coerce")
        non_nan = s.dropna()
        if not non_nan.empty and (non_nan == -1).all():
            all_minus1_cols.append(col)

    skip_expected_idx = df.columns.intersection(list(set(all_nan_cols).union(all_minus1_cols)))
    expected.loc[skip_expected_idx] = 0

    # --- Time-drift (duplicate-second) rows: mark extras only ---
    ts = pd.to_numeric(df["timestamp"], errors="coerce")
    ts_sec = np.floor(ts).astype("Int64")
    # extras only (keep first occurrence in each second)
    dup_mask = ts_sec.duplicated(keep="first").fillna(False)
    dup_extras_total = int(dup_mask.sum())

    # Component A: time-drift (affects all expected features equally in this file)
    missing_drift = pd.Series(0, index=cols, dtype="int64")
    if dup_extras_total > 0:
        add_mask = expected > 0
        missing_drift.loc[add_mask.index[add_mask]] = dup_extras_total

    # Remove drift rows BEFORE counting NaNs/electrical faults
    df_nodrift = df.loc[~dup_mask].copy()

    # --- Component B: NaNs after removing drift rows (only where expected>0) ---
    nan_counts_after = df_nodrift.isna().sum().reindex(cols).fillna(0).astype("int64")
    missing_nan = nan_counts_after.copy()
    missing_nan.loc[skip_expected_idx] = 0

    # --- Component C: electrical faults after removing drift rows ---
    elec_cols_present = [c for c in ELECTRICAL_COLS_CANON if c in cols]
    fault_count = 0
    if elec_cols_present:
        fault_mask = pd.Series(False, index=df_nodrift.index)
        for c in ("voltage", "frequency"):
            if c in df_nodrift.columns:
                s = pd.to_numeric(df_nodrift[c], errors="coerce")
                fault_mask |= (s == 0)
        fault_count = int(fault_mask.sum())
    missing_elec = pd.Series(0, index=cols, dtype="int64")
    if fault_count and elec_cols_present:
        add_elec = [c for c in elec_cols_present if expected.get(c, 0) > 0]
        missing_elec.loc[add_elec] = fault_count

    # --- Combine components for a "total" if you want one ---
    missing_total = (missing_drift + missing_nan + missing_elec).astype("int64")

    return {
        "expected": expected.astype("int64"),
        "missing_total": missing_total,
        "missing_components": {
            "drift": missing_drift.astype("int64"),
            "nan_after_drift": missing_nan.astype("int64"),
            "electrical_fault": missing_elec.astype("int64"),
        },
        "meta": {
            "dup_extras_total": dup_extras_total,
            "electrical_fault_rows": fault_count,
            "all_nan_cols": list(all_nan_cols),
            "all_minus1_cols": all_minus1_cols,
            "rows_after_drift": int(len(df_nodrift)),
        },
    }



def build_missing_report(laboratory: pd.DataFrame, denom: str = "expected_structural") -> dict:
    """
    Aggregates per-file two-pass missingness into per-machine and overall reports.

    Parameters
    ----------
    laboratory : pd.DataFrame
        Must contain columns: ['file_path', 'machine'].
    denom : {'expected_structural', 'expected_post_drift'}
        - 'expected_structural' : percentages use the original expected counts
          (rows per file unless the column is 100% NaN or all -1).
        - 'expected_post_drift' : percentages use expected minus the number of
          drift rows for that file (only for features with expected>0).

    Returns
    -------
    dict with:
      - per_machine_missing_total : DataFrame (index=machine, columns=features)
      - per_machine_expected      : DataFrame (index=machine, columns=features)
      - per_machine_components    : dict of DataFrames for each component
            {'drift': df, 'nan_after_drift': df, 'electrical_fault': df}
      - overall_feature_stats     : DataFrame index=feature with columns:
            ['expected', 'missing', 'missing_pct']  # where 'missing' is total
      - overall_feature_components: DataFrame with component columns:
            ['drift', 'nan_after_drift', 'electrical_fault', 'total']
      - overall_totals            : one-row DataFrame:
            ['expected_total', 'missing_total', 'missing_pct_total']
      - meta                      : info about rows/files seen and rules.
    """
    per_machine_missing_total_acc = defaultdict(lambda: None)
    per_machine_expected_acc = defaultdict(lambda: None)

    # component accumulators per machine
    comp_names = ["drift", "nan_after_drift", "electrical_fault"]
    per_machine_comp_acc = {c: defaultdict(lambda: None) for c in comp_names}

    rows_seen = defaultdict(int)
    files_seen = defaultdict(int)

    # also track per-machine, per-file drift counts so we can compute post-drift denominator
    per_machine_drift_rows_acc = defaultdict(int)

    for _, row in laboratory.iterrows():
        file_path = row["file_path"]
        machine = row["machine"]

        df = pd.read_csv(file_path)

        res = _per_file_feature_missing_and_expected_twopass(df)

        exp_s = res["expected"].astype("int64")
        miss_total_s = res["missing_total"].astype("int64")
        comps = {k: v.astype("int64") for k, v in res["missing_components"].items()}

        # accumulate totals per machine
        if per_machine_missing_total_acc[machine] is None:
            per_machine_missing_total_acc[machine] = miss_total_s.copy()
            per_machine_expected_acc[machine] = exp_s.copy()
        else:
            per_machine_missing_total_acc[machine] = per_machine_missing_total_acc[machine].add(miss_total_s, fill_value=0).astype("int64")
            per_machine_expected_acc[machine] = per_machine_expected_acc[machine].add(exp_s, fill_value=0).astype("int64")

        # accumulate components per machine
        for cname in comp_names:
            s = comps.get(cname, pd.Series(dtype="int64"))
            if s is None or s.empty:
                continue
            if per_machine_comp_acc[cname][machine] is None:
                per_machine_comp_acc[cname][machine] = s.copy()
            else:
                per_machine_comp_acc[cname][machine] = per_machine_comp_acc[cname][machine].add(s, fill_value=0).astype("int64")

        # drift rows (same for all features with expected>0 in this file)
        per_machine_drift_rows_acc[machine] += int(res["meta"].get("dup_extras_total", 0))

        rows_seen[machine] += len(df)
        files_seen[machine] += 1

    if not per_machine_missing_total_acc:
        return {
            "per_machine_missing_total": pd.DataFrame(),
            "per_machine_expected": pd.DataFrame(),
            "per_machine_components": {c: pd.DataFrame() for c in comp_names},
            "overall_feature_stats": pd.DataFrame(),
            "overall_feature_components": pd.DataFrame(),
            "overall_totals": pd.DataFrame(),
            "meta": {"rows_seen": {}, "files_seen": {}},
        }

    # Union of columns across machines
    all_cols = sorted(set().union(*[s.index for s in per_machine_missing_total_acc.values()]))

    machines_sorted = sorted(per_machine_missing_total_acc.keys())
    per_machine_missing_total = pd.DataFrame(index=machines_sorted, columns=all_cols, dtype="int64").fillna(0)
    per_machine_expected = pd.DataFrame(index=machines_sorted, columns=all_cols, dtype="int64").fillna(0)

    # Fill main per-machine tables
    for m in machines_sorted:
        if per_machine_missing_total_acc[m] is not None:
            per_machine_missing_total.loc[m, per_machine_missing_total_acc[m].index] = per_machine_missing_total_acc[m]
        if per_machine_expected_acc[m] is not None:
            per_machine_expected.loc[m, per_machine_expected_acc[m].index] = per_machine_expected_acc[m]

    # Build per-machine component tables
    per_machine_components = {}
    for cname in comp_names:
        df_comp = pd.DataFrame(index=machines_sorted, columns=all_cols, dtype="int64").fillna(0)
        for m in machines_sorted:
            acc = per_machine_comp_acc[cname][m]
            if acc is not None:
                df_comp.loc[m, acc.index] = acc
        per_machine_components[cname] = df_comp

    # Overall sums per feature
    overall_missing_total = per_machine_missing_total.sum(axis=0).astype("int64")
    overall_expected_struct = per_machine_expected.sum(axis=0).astype("int64")

    # Overall components
    overall_comp = {}
    for cname in comp_names:
        overall_comp[cname] = per_machine_components[cname].sum(axis=0).astype("int64")
    overall_comp_df = pd.DataFrame(overall_comp)
    overall_comp_df["total"] = overall_comp_df.sum(axis=1).astype("int64")

    # Choose denominator
    if denom == "expected_structural":
        overall_expected_for_pct = overall_expected_struct.copy()
    elif denom == "expected_post_drift":
        # Subtract drift rows only where feature was expected (>0).
        # We approximate by subtracting overall_comp['drift'] from expected_struct,
        # clipped at zero.
        overall_expected_for_pct = (overall_expected_struct - overall_comp_df["drift"]).clip(lower=0)
    else:
        raise ValueError("denom must be one of {'expected_structural', 'expected_post_drift'}")

    # Safe percentage
    with np.errstate(divide="ignore", invalid="ignore"):
        overall_pct = (overall_missing_total / overall_expected_for_pct.replace(0, np.nan)) * 100.0
    overall_pct = overall_pct.fillna(0.0)

    overall_feature_stats = pd.DataFrame({
        "expected": overall_expected_for_pct.astype("int64"),
        "missing": overall_missing_total,
        "missing_pct": overall_pct.round(2),
    }).sort_values("missing_pct", ascending=False)

    # Grand totals
    expected_total = int(overall_expected_for_pct.sum())
    missing_total  = int(overall_missing_total.sum())
    missing_pct_total = round((missing_total / expected_total) * 100.0, 2) if expected_total > 0 else 0.0

    overall_totals = pd.DataFrame([{
        "expected_total": expected_total,
        "missing_total": missing_total,
        "missing_pct_total": missing_pct_total,
    }])

    meta = {
        "rows_seen": dict(rows_seen),
        "files_seen": dict(files_seen),
        "electrical_features_considered": ELECTRICAL_COLS_CANON,
        "rules": {
            "calculation_passes": [
                "A) Identify duplicate-second extras (time drift), drop extras.",
                "B) Count NaNs on remaining rows.",
                "C) Count electrical faults (e.g., voltage==0 or frequency==0) on remaining rows.",
            ],
            "components": ["drift", "nan_after_drift", "electrical_fault"],
            "minus_one_handling": "-1 values are ignored (not treated as missing).",
            "expected_definition_structural": "per-file expected = number of rows unless the column is 100% NaN or all -1 (then 0).",
            "denominator_mode": denom,
        },
    }

    return {
        "per_machine_missing_total": per_machine_missing_total,
        "per_machine_expected": per_machine_expected,
        "per_machine_components": per_machine_components,
        "overall_feature_stats": overall_feature_stats,
        "overall_feature_components": overall_comp_df.sort_index(),
        "overall_totals": overall_totals,
        "meta": meta,
    }


<h3 style="
    font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
    color:#D4495e;
    margin-top:15px;
    font-weight:600;
">
    Summary of missing data points per recorded feature across laboratory washing cycles within existing data columns.
</h3>

In [ ]:
# Filter your master table first
laboratory = aggregated[
    (aggregated['environment'] == 'laboratory') &
    (aggregated['type'] == 'wm') # wm = washing machine
]


report = build_missing_report(laboratory)

# Per-machine missing and expected tables
print(report["per_machine_missing_total"])

# Overall per-feature stats (expected, missing, missing %)
print(report["overall_feature_stats"])

# One-line overall totals across all features
print(report["overall_totals"])


<h3 style="
    font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
    color:#D4495e;
    margin-top:15px;
    font-weight:600;
">
    Summary of missing data points across all features in dryer machine measurements. All missing entries are caused
by time drift in the recording process, which results in two samples being assigned the same timestamp (second) and one of
them being excluded. The “Total” row aggregates the counts across all features.
</h3>

In [ ]:
# Filter your master table first
laboratory = aggregated[
    (aggregated['environment'] == 'laboratory') &
    (aggregated['type'] == 'dm') # dm = dryer machine
]


report = build_missing_report(laboratory)

# Per-machine missing and expected tables
# print(report["per_machine_missing"])
print(report["per_machine_expected"])

# Overall per-feature stats (expected, missing, missing %)
print(report["overall_feature_stats"])

# One-line overall totals across all features
print(report["overall_totals"])


<h3 style="
    font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
    color:#D4495e;
    margin-top:15px;
    font-weight:600;
">
    Missing washing machine and dryer data per accelerometer axis. Missing samples are due to row-level dropouts
affecting all channels simultaneously
</h3>

In [ ]:
# Filter your master table first
aggregated_acc = pd.read_csv("aggregated_data_acc.csv")

acc_lab = aggregated_acc[
    (aggregated_acc['environment'] == 'laboratory')
]

In [ ]:
import pandas as pd
import numpy as np

FS = 200  # assumed sampling rate (Hz)
AXES = ["back.x", "back.y", "back.z", "side.x", "side.y", "side.z", "top.x", "top.y", "top.z"]

# Accumulators
axis_missing = {ax: 0 for ax in AXES}     # total missing samples per axis across files
axis_expected = {ax: 0 for ax in AXES}    # total expected samples per axis across files
files_processed = 0

for _, row in acc_lab.iterrows():
    file_path = row["file_path"]

    try:
        df = pd.read_parquet(file_path)
    except Exception:
        # if unreadable, skip (or count as fully missing if you prefer)
        continue

    if df.empty or "timestamp" not in df.columns:
        continue

    # Ensure timestamp is datetime and sorted
    ts = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
    df = df.assign(timestamp=ts).dropna(subset=["timestamp"]).sort_values("timestamp")

    if df.empty:
        continue

    files_processed += 1

    t0 = df["timestamp"].iloc[0]
    t1 = df["timestamp"].iloc[-1]
    duration_s = (t1 - t0).total_seconds()

    # Expected rows from t0..t1 inclusive at FS Hz (approx)
    # Using +1 accounts for both endpoints in an evenly sampled series.
    expected_rows = int(round(duration_s * FS)) + 1
    actual_rows = len(df)
    time_missing = max(0, expected_rows - actual_rows)

    # For each axis: missing = time_missing + NaNs in that axis
    for ax in AXES:
        if ax in df.columns:
            nan_missing = int(df[ax].isna().sum())
            axis_missing[ax] += time_missing + nan_missing
            axis_expected[ax] += expected_rows
        else:
            # Column not present: all expected are missing for this file
            axis_missing[ax] += expected_rows
            axis_expected[ax] += expected_rows

# Build summary table
summary = (
    pd.DataFrame({
        "axis": AXES,
        "expected": [axis_expected[a] for a in AXES],
        "missing":  [axis_missing[a]  for a in AXES],
    })
    .assign(missing_pct=lambda d: (100.0 * d["missing"] / d["expected"]).round(2))
    .sort_values("missing_pct", ascending=False)
    .reset_index(drop=True)
)

print(f"Files processed: {files_processed}")
print(summary)

<h3 style="
    font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
    color:#D4495e;
    margin-top:15px;
    font-weight:600;
">
    Statistics of missing audio files of washing machines
</h3>

In [ ]:
import pandas as pd
# Filter your master table first
laboratory = aggregated[
    (aggregated['environment'] == 'laboratory') &
    (aggregated['type'] == 'wm') # dm = dryer machine
]

In [ ]:
count_audio_files = (laboratory["has_audio"] == 1.0).sum()

print(f"Number of files with audio: {count_audio_files} out of {len(laboratory)}, Missing {len(laboratory) - count_audio_files}, Missing %: {100 * (1 - count_audio_files / len(laboratory)):.2f}%")

<h3 style="
    font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
    color:#D4495e;
    margin-top:15px;
    font-weight:600;
">
    Statistics of accelerometer files of washing machines
</h3>

In [ ]:
import pandas as pd
# Filter your master table first
laboratory = aggregated[
    (aggregated['environment'] == 'laboratory') &
    (aggregated['type'] == 'wm') # dm = dryer machine
]

# Filter your master table first
aggregated_acc = pd.read_csv("aggregated_data_acc.csv")

acc_lab = aggregated_acc[
    (aggregated_acc['environment'] == 'laboratory') &
    (aggregated_acc['type'] == 'wm') # dm = dryer machine
]

In [ ]:
empty_count = 0
for idx, r in acc_lab.iterrows():
    file_path = r["file_path"]

    try:
        df = pd.read_parquet(file_path)

        if df.empty:
            # print(f"❌ Empty file: {file_path}")
            empty_count += 1
        else:
            # print(f"✅ Non-empty file: {file_path} (rows: {len(df)})")
            continue

    except Exception as e:
        print(f"⚠️ Error reading file: {file_path}")
        print(f"   {e}")

# Count missing accelerometer files
print(f'Expected {len(laboratory)}, Found {len(acc_lab)}, Missing {len(laboratory)-len(acc_lab)+empty_count}, Mising % {(len(laboratory)-len(acc_lab)+empty_count)/len(laboratory)*100:.2f}%')

<h3 style="
    font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
    color:#D4495e;
    margin-top:15px;
    font-weight:600;
">
    Summary of missing data (entire column) across laboratory washing machine cycles (N= 643).
</h3>

In [ ]:
import pandas as pd

EXPECTED_COLS = [
    "timestamp", "current", "frequency", "power", "power_factor", "voltage",
    "water_inlet", "water_outlet", "drum_door_temp", "drum_temperature",
    "water_temperature_in", "water_temperature_out",
    "ambient_temperature_1", "ambient_temperature_2",
    "ambient_humidity_1", "ambient_humidity_2",
    "pressure"
]

file_count = len(laboratory)   # total number of files
missing_counts = {c: 0 for c in EXPECTED_COLS}

def is_all_minus_one(series: pd.Series) -> bool:
    """True if the column exists but all values are -1 (ignoring NaNs)."""
    s = pd.to_numeric(series, errors="coerce").dropna()
    return not s.empty and (s == -1).all()

for _, row in laboratory.iterrows():
    file_path = row["file_path"]
    try:
        df = pd.read_csv(file_path)
    except Exception:
        # If unreadable, mark everything as missing for this file
        for col in EXPECTED_COLS:
            missing_counts[col] += 1
        continue

    for col in EXPECTED_COLS:
        if col not in df.columns:
            missing_counts[col] += 1
        else:
            col_series = pd.to_numeric(df[col], errors="coerce").dropna()

            # General missing rule
            missing_flag = is_all_minus_one(col_series)

            # Special case: water_inlet
            if col == "water_inlet":
                if col_series.sum() == 0:
                    missing_flag = True

            # Special case: water_outlet
            elif col == "water_outlet":
                if col_series.sum() < 1000:
                    missing_flag = True

            if missing_flag:
                missing_counts[col] += 1

# Build summary table
summary = (
    pd.DataFrame({
        "expected": file_count,
        "missing": pd.Series(missing_counts),
    })
    .assign(missing_pct=lambda d: (d["missing"] / d["expected"] * 100).round(2))
    .sort_values("missing_pct", ascending=False)
)

print(summary)